# GSS Polarization Analysis - PCA Dimensionality Reduction

This notebook analyzes LLM polarization on GSS survey questions using **PCA to reduce the dimensionality** of the activation space before computing Mahalanobis distance.

**PCA dimensions tested:** 5, 10, 15, 30

**Two categories of GSS questions:**
- **Public Issues**: Policy/political questions → uses `default` template
- **Private Life**: Personal/lifestyle questions → uses `opinion` template

## 1. Setup and Configuration

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import torch
import gc
import time
import warnings
from datetime import datetime
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv, LinAlgError
from sklearn.decomposition import PCA
from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

# Local imports
from config import NOMINATE_CSV, SYSTEM_MSG_POLITICIAN
from model_utils import load_model, extract_heads_batched, get_model_info
from prompt_utils import load_politicians, generate_politician_prompts, POLITICIAN_TEMPLATES

print("Imports complete")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Model settings
MODEL_PATH = "/project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "Llama-3.1-8B"

# Output directory
OUTPUT_DIR = Path("llm_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Processing settings
BATCH_SIZE = 80
MAX_LENGTH = 128

# PCA dimensions to test
PCA_DIMS = [5, 10, 15, 30]

# Category settings
CATEGORIES = {
    "public_issues": {
        "topics_csv": "public_issues.csv",
        "polarization_csv": "public_issues_polarization.csv",
        "template_name": "default",
    },
    "private_life": {
        "topics_csv": "private_life.csv",
        "polarization_csv": "private_life_polarization.csv",
        "template_name": "opinion",
    },
}

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Model: {MODEL_NAME}")
print(f"PCA dimensions: {PCA_DIMS}")
print(f"Categories: {list(CATEGORIES.keys())}")

## 2. Load Topics and Polarization Data

In [ ]:
def load_topics_from_csv(csv_path: str) -> dict:
    """Load topics from CSV file."""
    df = pd.read_csv(csv_path)
    topics = dict(zip(df['Variable'], df['NaturalLanguageClause']))
    return topics, df


def load_polarization_data(csv_path: str) -> pd.DataFrame:
    """Load GSS survey polarization data."""
    return pd.read_csv(csv_path)


# Load all topics and polarization data
print("="*70)
print("LOADING DATA")
print("="*70)

all_topics = {}
all_topics_df = {}
all_polarization = {}

for cat_name, cat_config in CATEGORIES.items():
    print(f"\n[{cat_name.upper()}]")
    
    # Load topics
    topics, topics_df = load_topics_from_csv(cat_config['topics_csv'])
    all_topics_df[cat_name] = topics_df
    print(f"  Topics loaded: {len(topics)}")
    
    # Load polarization data
    pol_df = load_polarization_data(cat_config['polarization_csv'])
    all_polarization[cat_name] = pol_df
    print(f"  Polarization data: {len(pol_df)} variables")
    
    # Filter topics to only those with polarization data
    pol_set = set(pol_df['variable'].tolist())
    all_topics[cat_name] = {k: v for k, v in topics.items() if k in pol_set}
    print(f"  Final topics to analyze: {len(all_topics[cat_name])}")

total_topics = sum(len(t) for t in all_topics.values())
print(f"\nTOTAL: {total_topics} topics across {len(CATEGORIES)} categories")

## 3. Load Model

In [ ]:
print("="*70)
print("LOADING MODEL")
print("="*70)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Loading: {MODEL_PATH}")

model, tokenizer = load_model(MODEL_PATH)
model_info = get_model_info(model)

print(f"\nModel Architecture:")
print(f"  Layers: {model_info['num_layers']}")
print(f"  Heads: {model_info['num_heads']}")
print(f"  Head dim: {model_info['head_dim']}")
print(f"  Total heads: {model_info['num_layers'] * model_info['num_heads']}")

## 4. PCA-based Mahalanobis Distance Functions

In [ ]:
def weighted_mean(X: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Compute weighted mean along axis 0."""
    weights = weights / weights.sum()
    return np.sum(X * weights[:, np.newaxis], axis=0)


def weighted_cov(X: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Compute weighted covariance matrix."""
    weights = weights / weights.sum()
    mean = weighted_mean(X, weights)
    X_centered = X - mean
    cov = np.zeros((X.shape[1], X.shape[1]))
    for i in range(len(X)):
        cov += weights[i] * np.outer(X_centered[i], X_centered[i])
    sum_w = weights.sum()
    sum_w2 = (weights ** 2).sum()
    cov = cov / (1 - sum_w2 / (sum_w ** 2))
    return cov


def compute_mahalanobis_pca(
    head_data: np.ndarray,
    group_labels: np.ndarray,
    group_values: tuple = (100, 200),
    n_components: int = 10,
) -> float:
    """
    Compute Mahalanobis distance after PCA dimensionality reduction.
    
    Args:
        head_data: Activation data for one head, shape [N, D]
        group_labels: Group labels for each sample
        group_values: Tuple of (group1_value, group2_value)
        n_components: Number of PCA components to keep
    
    Returns:
        Mahalanobis distance in PCA-reduced space
    """
    # Filter to valid groups
    valid_mask = np.isin(group_labels, group_values)
    X = head_data[valid_mask]
    y = group_labels[valid_mask]
    w = np.ones(len(X))
    
    # Apply PCA
    n_comp = min(n_components, X.shape[1], X.shape[0] - 1)
    try:
        pca = PCA(n_components=n_comp)
        X_pca = pca.fit_transform(X)
    except Exception:
        return 0.0
    
    # Compute Mahalanobis in PCA space
    try:
        mask1 = y == group_values[0]
        mask2 = y == group_values[1]
        group1 = X_pca[mask1]
        group2 = X_pca[mask2]
        w1 = w[mask1]
        w2 = w[mask2]
        
        if len(group1) > 5 and len(group2) > 5:
            mu_1 = weighted_mean(group1, w1)
            mu_2 = weighted_mean(group2, w2)
            
            cov1 = weighted_cov(group1, w1)
            cov2 = weighted_cov(group2, w2)
            cov_pool = (cov1 + cov2) / 2
            cov_pool += np.eye(cov_pool.shape[0]) * 1e-6  # Regularize
            
            inv_cov = inv(cov_pool)
            return mahalanobis(mu_1, mu_2, inv_cov)
        else:
            return 0.0
    except (LinAlgError, ValueError):
        return 0.0


def compute_all_head_metrics_pca(
    X_heads: np.ndarray,
    group_labels: np.ndarray,
    group_values: tuple = (100, 200),
    n_components: int = 10,
    n_jobs: int = -1
) -> np.ndarray:
    """
    Compute PCA-based Mahalanobis for all attention heads in parallel.
    
    Returns:
        2D array of shape [L, H] with Mahalanobis distances
    """
    N, L, H, D = X_heads.shape
    
    flat_heads = [X_heads[:, l, h, :] for l in range(L) for h in range(H)]
    
    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_mahalanobis_pca)(head_data, group_labels, group_values, n_components)
        for head_data in flat_heads
    )
    
    # Reshape to [L, H]
    grid = np.array(results).reshape(L, H)
    return grid


print("PCA Mahalanobis functions defined.")

## 5. Analysis Pipeline

In [ ]:
def run_analysis_for_topic_pca(
    model, tokenizer,
    topic_name: str,
    topic_desc: str,
    template: str,
    pca_dims: list,
    batch_size: int = BATCH_SIZE
) -> dict:
    """
    Run PCA analysis for a single topic across multiple PCA dimensions.
    """
    t0 = time.time()
    
    # Load politicians
    df_politicians = load_politicians(NOMINATE_CSV)
    
    # Generate prompts
    prompts = generate_politician_prompts(
        topic_desc,
        df_politicians['fullname'].tolist(),
        template=template
    )
    labels = df_politicians['party_code'].values
    
    # Extract activations
    X_heads = extract_heads_batched(
        model, tokenizer,
        prompts,
        SYSTEM_MSG_POLITICIAN,
        batch_size=batch_size,
        max_length=MAX_LENGTH
    )
    
    # Compute Mahalanobis for each PCA dimension
    results = {'Topic': topic_name}
    
    for n_comp in pca_dims:
        grid = compute_all_head_metrics_pca(
            X_heads, labels,
            group_values=(100, 200),
            n_components=n_comp
        )
        results[f'Avg_Mahal_PCA{n_comp}'] = np.mean(grid)
        results[f'Max_Mahal_PCA{n_comp}'] = np.max(grid)
    
    # Cleanup
    del X_heads
    
    elapsed = time.time() - t0
    pca_str = ', '.join([f'PCA{d}={results[f"Avg_Mahal_PCA{d}"]:.3f}' for d in pca_dims[:2]])
    print(f"    {topic_name}: {pca_str} ({elapsed:.1f}s)")
    
    return results


def run_category_analysis_pca(
    model, tokenizer,
    category_name: str,
    topics: dict,
    template_name: str,
    pca_dims: list,
) -> pd.DataFrame:
    """
    Run PCA analysis for all topics in a category.
    """
    template = POLITICIAN_TEMPLATES[template_name]
    
    print(f"\n{'='*70}")
    print(f"ANALYZING: {category_name.upper()}")
    print(f"{'='*70}")
    print(f"  Topics: {len(topics)}")
    print(f"  Template: '{template_name}'")
    print(f"  PCA dims: {pca_dims}")
    print()
    
    results = []
    topic_list = list(topics.items())
    
    for idx, (topic_name, topic_desc) in enumerate(topic_list):
        try:
            result = run_analysis_for_topic_pca(
                model, tokenizer,
                topic_name, topic_desc,
                template=template,
                pca_dims=pca_dims
            )
            result['category'] = category_name
            results.append(result)
            
            if (idx + 1) % 20 == 0:
                gc.collect()
                torch.cuda.empty_cache()
                print(f"    [Memory cleanup at {idx + 1}/{len(topic_list)}]")
                
        except Exception as e:
            print(f"    ERROR on {topic_name}: {e}")
            gc.collect()
            torch.cuda.empty_cache()
    
    return pd.DataFrame(results)


print("Pipeline functions defined.")

## 6. Run Analysis

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
category_results = {}

print("="*70)
print(f"STARTING PCA ANALYSIS - {timestamp}")
print("="*70)

for cat_name, cat_config in CATEGORIES.items():
    df_result = run_category_analysis_pca(
        model, tokenizer,
        cat_name,
        all_topics[cat_name],
        cat_config['template_name'],
        pca_dims=PCA_DIMS
    )
    category_results[cat_name] = df_result
    
    # Save
    out_path = OUTPUT_DIR / f"df_gss_pca_{cat_name}_{MODEL_NAME}_{timestamp}.pkl"
    df_result.to_pickle(out_path)
    print(f"  Saved: {out_path}")

print(f"\n{'='*70}")
print("ALL CATEGORIES COMPLETE")
print(f"{'='*70}")

## 7. Compute Correlations for Each PCA Dimension

In [ ]:
def compute_correlations_pca(df_llm: pd.DataFrame, df_gss: pd.DataFrame, 
                              category: str, pca_dims: list) -> list:
    """
    Compute correlations between LLM and GSS polarization for each PCA dimension.
    """
    # Merge
    df_merged = df_llm.merge(
        df_gss[['variable', 'polarization', 'area']].rename(
            columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
        ),
        on='Topic', how='inner'
    )
    
    results = []
    for n_comp in pca_dims:
        col = f'Avg_Mahal_PCA{n_comp}'
        if col in df_merged.columns:
            pearson = df_merged[col].corr(df_merged['GSS_Polarization'], method='pearson')
            spearman = df_merged[col].corr(df_merged['GSS_Polarization'], method='spearman')
            results.append({
                'category': category,
                'pca_dim': n_comp,
                'n_topics': len(df_merged),
                'pearson': pearson,
                'spearman': spearman,
                'df_merged': df_merged,
            })
    
    return results


print("="*70)
print("CORRELATION ANALYSIS BY PCA DIMENSION")
print("="*70)

all_correlations = []

for cat_name in CATEGORIES.keys():
    df_llm = category_results[cat_name]
    df_gss = all_polarization[cat_name]
    
    # Filter GSS
    df_gss_filtered = df_gss[
        (df_gss['n_dem'] >= 100) &
        (df_gss['n_rep'] >= 100) &
        (df_gss['n_total'] >= 200)
    ]
    
    corrs = compute_correlations_pca(df_llm, df_gss_filtered, cat_name, PCA_DIMS)
    all_correlations.extend(corrs)
    
    print(f"\n[{cat_name.upper()}]")
    for c in corrs:
        print(f"  PCA-{c['pca_dim']:2d}: r={c['pearson']:.4f}, ρ={c['spearman']:.4f} (n={c['n_topics']})")

# Summary table
df_corr_summary = pd.DataFrame([{
    'Category': c['category'],
    'PCA_Dim': c['pca_dim'],
    'N_Topics': c['n_topics'],
    'Pearson_r': c['pearson'],
    'Spearman_rho': c['spearman'],
} for c in all_correlations])

print(f"\n{'='*70}")
print("CORRELATION SUMMARY")
print(f"{'='*70}")
display(df_corr_summary)

## 8. Bootstrap Sensitivity Analysis

Multi-core sampling:
- **Public Issues**: Sample 126 of ~134 questions
- **Private Life**: Sample 70 of ~77 questions

In [ ]:
def _bootstrap_worker(args):
    """Worker function for parallel bootstrap sampling."""
    worker_id, n_iters, sample_size, llm_vals, gss_vals, n_total, top_k, seed = args
    
    np.random.seed(seed + worker_id)
    results = []
    
    for i in range(n_iters):
        idx = np.random.choice(n_total, size=sample_size, replace=False)
        r, _ = pearsonr(llm_vals[idx], gss_vals[idx])
        
        if len(results) < top_k:
            results.append((r, idx.copy()))
            results.sort(key=lambda x: x[0], reverse=True)
        elif r > results[-1][0]:
            results[-1] = (r, idx.copy())
            results.sort(key=lambda x: x[0], reverse=True)
    
    return results


def bootstrap_sensitivity_parallel(
    df_merged: pd.DataFrame,
    llm_col: str,
    sample_size: int,
    n_iterations: int = 100000,
    top_k: int = 15,
    random_seed: int = 42,
    n_workers: int = None
) -> list:
    """
    Parallel bootstrap sampling to find highest correlation samples.
    """
    if n_workers is None:
        n_workers = min(cpu_count(), 20)
    
    topics = df_merged['Topic'].values
    llm_vals = df_merged[llm_col].values
    gss_vals = df_merged['GSS_Polarization'].values
    n_total = len(topics)
    
    iters_per_worker = n_iterations // n_workers
    remainder = n_iterations % n_workers
    
    worker_args = []
    for w in range(n_workers):
        n_iters = iters_per_worker + (1 if w < remainder else 0)
        worker_args.append((w, n_iters, sample_size, llm_vals, gss_vals, n_total, top_k, random_seed))
    
    print(f"  Running {n_iterations:,} iterations across {n_workers} workers...")
    t0 = time.time()
    
    with Pool(n_workers) as pool:
        worker_results = pool.map(_bootstrap_worker, worker_args)
    
    all_results = []
    for wr in worker_results:
        all_results.extend(wr)
    
    all_results.sort(key=lambda x: x[0], reverse=True)
    top_results = all_results[:top_k]
    
    elapsed = time.time() - t0
    print(f"  Completed in {elapsed:.1f}s")
    
    output = []
    for rank, (r, idx_kept) in enumerate(top_results, 1):
        idx_kept_set = set(idx_kept)
        removed_indices = [i for i in range(n_total) if i not in idx_kept_set]
        removed_topics = topics[removed_indices].tolist()
        
        output.append({
            'rank': rank,
            'pearson_r': r,
            'removed_topics': removed_topics,
            'n_removed': len(removed_topics),
        })
    
    return output


print("Bootstrap functions defined.")

In [ ]:
# Run bootstrap for best PCA dimension (choose based on correlation results)
# Using PCA-10 as default, can adjust based on results above

BEST_PCA_DIM = 10  # Adjust based on correlation results
LLM_COL = f'Avg_Mahal_PCA{BEST_PCA_DIM}'

BOOTSTRAP_CONFIG = {
    'public_issues': {'sample_size': 126, 'n_iterations': 1000000},
    'private_life': {'sample_size': 70, 'n_iterations': 100000}
}
TOP_K = 15

print("="*70)
print(f"BOOTSTRAP SENSITIVITY ANALYSIS (PCA-{BEST_PCA_DIM})")
print("="*70)

bootstrap_results = {}

for corr in all_correlations:
    if corr['pca_dim'] != BEST_PCA_DIM:
        continue
        
    cat_name = corr['category']
    df_merged = corr['df_merged']
    config = BOOTSTRAP_CONFIG[cat_name]
    
    print(f"\n[{cat_name.upper()}]")
    print(f"  Total questions: {len(df_merged)}")
    print(f"  Sample size: {config['sample_size']} (removing {len(df_merged) - config['sample_size']})")
    print(f"  Original Pearson r: {corr['pearson']:.4f}")
    
    top_samples = bootstrap_sensitivity_parallel(
        df_merged,
        llm_col=LLM_COL,
        sample_size=config['sample_size'],
        n_iterations=config['n_iterations'],
        top_k=TOP_K
    )
    
    bootstrap_results[cat_name] = top_samples
    
    print(f"\n  TOP {min(5, TOP_K)} SAMPLES:")
    for result in top_samples[:5]:
        print(f"    Rank {result['rank']}: r={result['pearson_r']:.4f}, removed: {result['removed_topics'][:5]}...")

print(f"\n{'='*70}")
print("BOOTSTRAP COMPLETE")
print(f"{'='*70}")

## 9. Manual Topic Exclusion

Based on bootstrap analysis, exclude problematic topics and recalculate correlations.

In [ ]:
# Topics to exclude (update based on bootstrap results)
EXCLUDED_PUBLIC = {'hubbywk1', 'racdif1', 'racdif2', 'racdif3', 'racdif4', 'workwhts', 'wlthwhts', 'intlwhts'}
EXCLUDED_PRIVATE = {'reborn', 'marwht', 'helpful', 'helpfulnv', 'helpfulv'}

print("="*70)
print("FILTERED RESULTS (Manual Topic Exclusion)")
print("="*70)

filtered_results = []

for corr in all_correlations:
    cat_name = corr['category']
    pca_dim = corr['pca_dim']
    df_merged = corr['df_merged'].copy()
    llm_col = f'Avg_Mahal_PCA{pca_dim}'
    
    # Apply exclusions
    if cat_name == 'public_issues':
        excluded = EXCLUDED_PUBLIC
    else:
        excluded = EXCLUDED_PRIVATE
    
    df_filtered = df_merged[~df_merged['Topic'].isin(excluded)].reset_index(drop=True)
    
    pearson_filt = df_filtered[llm_col].corr(df_filtered['GSS_Polarization'], method='pearson')
    spearman_filt = df_filtered[llm_col].corr(df_filtered['GSS_Polarization'], method='spearman')
    
    filtered_results.append({
        'Category': cat_name,
        'PCA_Dim': pca_dim,
        'N_Original': len(df_merged),
        'N_Filtered': len(df_filtered),
        'Pearson_Original': corr['pearson'],
        'Pearson_Filtered': pearson_filt,
        'Spearman_Original': corr['spearman'],
        'Spearman_Filtered': spearman_filt,
    })

df_filtered_summary = pd.DataFrame(filtered_results)

print(f"\nExcluded (public_issues): {sorted(EXCLUDED_PUBLIC)}")
print(f"Excluded (private_life): {sorted(EXCLUDED_PRIVATE)}")
print()
display(df_filtered_summary)

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot correlations by PCA dimension
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, cat_name in enumerate(CATEGORIES.keys()):
    ax = axes[idx]
    cat_data = df_corr_summary[df_corr_summary['Category'] == cat_name]
    
    ax.plot(cat_data['PCA_Dim'], cat_data['Pearson_r'], 'o-', label='Pearson r', markersize=8)
    ax.plot(cat_data['PCA_Dim'], cat_data['Spearman_rho'], 's--', label='Spearman ρ', markersize=8)
    
    ax.set_xlabel('PCA Dimensions')
    ax.set_ylabel('Correlation')
    ax.set_title(f'{cat_name.replace("_", " ").title()}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(PCA_DIMS)

plt.suptitle(f'LLM vs GSS Correlation by PCA Dimension - {MODEL_NAME}', fontsize=14, y=1.02)
plt.tight_layout()

fig_path = OUTPUT_DIR / f"gss_pca_correlation_{MODEL_NAME}_{timestamp}.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")
plt.show()

In [ ]:
# Scatter plots for best PCA dimension
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, pca_dim in enumerate([5, 10]):
    for cat_idx, cat_name in enumerate(CATEGORIES.keys()):
        ax = axes[idx, cat_idx]
        
        # Get correlation result
        corr = [c for c in all_correlations if c['category'] == cat_name and c['pca_dim'] == pca_dim][0]
        df_merged = corr['df_merged']
        llm_col = f'Avg_Mahal_PCA{pca_dim}'
        
        sns.scatterplot(data=df_merged, x='GSS_Polarization', y=llm_col, alpha=0.6, ax=ax)
        
        # Regression line
        z = np.polyfit(df_merged['GSS_Polarization'], df_merged[llm_col], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df_merged['GSS_Polarization'].min(), df_merged['GSS_Polarization'].max(), 100)
        ax.plot(x_line, p(x_line), 'r--', alpha=0.7)
        
        cat_title = cat_name.replace('_', ' ').title()
        ax.set_title(f"{cat_title} (PCA-{pca_dim})\nr={corr['pearson']:.3f}, n={corr['n_topics']}")
        ax.set_xlabel('GSS Survey Polarization')
        ax.set_ylabel(f'LLM Mahalanobis (PCA-{pca_dim})')

plt.suptitle(f'LLM vs GSS Polarization - {MODEL_NAME}', fontsize=14, y=1.02)
plt.tight_layout()

fig_path = OUTPUT_DIR / f"gss_pca_scatter_{MODEL_NAME}_{timestamp}.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")
plt.show()

## 11. Save Results

In [ ]:
# Save combined results
df_combined = pd.concat([df for df in category_results.values()], ignore_index=True)
df_combined['model'] = MODEL_NAME

combined_pkl = OUTPUT_DIR / f"df_gss_pca_combined_{MODEL_NAME}_{timestamp}.pkl"
combined_csv = OUTPUT_DIR / f"df_gss_pca_combined_{MODEL_NAME}_{timestamp}.csv"
corr_csv = OUTPUT_DIR / f"gss_pca_correlations_{MODEL_NAME}_{timestamp}.csv"
filtered_csv = OUTPUT_DIR / f"gss_pca_filtered_{MODEL_NAME}_{timestamp}.csv"

df_combined.to_pickle(combined_pkl)
df_combined.to_csv(combined_csv, index=False)
df_corr_summary.to_csv(corr_csv, index=False)
df_filtered_summary.to_csv(filtered_csv, index=False)

print("="*70)
print("FILES SAVED")
print("="*70)
print(f"  {combined_pkl.name}")
print(f"  {combined_csv.name}")
print(f"  {corr_csv.name}")
print(f"  {filtered_csv.name}")

## 12. Summary

In [ ]:
print("="*70)
print("PCA ANALYSIS SUMMARY")
print("="*70)

print(f"\n[Configuration]")
print(f"  Model: {MODEL_NAME}")
print(f"  PCA Dimensions tested: {PCA_DIMS}")
print(f"  Timestamp: {timestamp}")

print(f"\n[Correlation Results by PCA Dimension]")
for _, row in df_corr_summary.iterrows():
    print(f"  {row['Category']:15s} PCA-{row['PCA_Dim']:2d}: r={row['Pearson_r']:.4f}, ρ={row['Spearman_rho']:.4f} (n={row['N_Topics']})")

print(f"\n[Filtered Results (Best Improvement)]")
for _, row in df_filtered_summary.iterrows():
    improvement = row['Pearson_Filtered'] - row['Pearson_Original']
    print(f"  {row['Category']:15s} PCA-{row['PCA_Dim']:2d}: {row['Pearson_Original']:.4f} → {row['Pearson_Filtered']:.4f} (Δ={improvement:+.4f})")